# Visuall Attendance System 

In [1]:
import cv2
import face_recognition
import os
import csv
import pandas as pd
from datetime import datetime

In [ ]:

# ========================
# 1. SETUP employee DATABASE
# ========================

def create_employee_database():
    """Create a CSV file with sample employee data if it doesn't exist"""
    if not os.path.exists("ict_employees.csv"):
        with open("employees.csv", mode='w', newline='') as file:
            writer = csv.writer(file)
            writer.writerow(["RollNo", "Name", "ImagePath"])
            writer.writerow([1, "Simret Mequanent", "employees/simret.JPG"])
            writer.writerow([2, "Tigist Teshome", "employees/tigist.JPG"])
            writer.writerow([3, "Mulatu Wondimu", "employees/mulatu.JPG"])
            writer.writerow([4, "Milkias Elias", "employees/milkias.JPG"])
            writer.writerow([5, "Kibru Bogale", "employees/kibru.JPG"])
            writer.writerow([5, "Ibsa Nuru", "employees/ibsa.JPG"])
            writer.writerow([5, "Fantahun Bogale", "employees/Fanta.JPG"])
            
        print("Created employees.csv with sample data")

def load_employee_data():
    """Load employee data and face encodings"""
    employees = []
    with open("employees.csv", mode='r') as file:
        reader = csv.DictReader(file)
        for row in reader:
            # Load employee image and generate encoding
            image_path = row["ImagePath"]
            if not os.path.exists(image_path):
                print(f"Warning: Image not found for {row['Name']} ({image_path})")
                continue
                
            image = face_recognition.load_image_file(image_path)
            encodings = face_recognition.face_encodings(image)
            
            if len(encodings) > 0:
                employees.append({
                    "RollNo": row["RollNo"],
                    "Name": row["Name"],
                    "Encoding": encodings[0]
                })
            else:
                print(f"Warning: No face found in {image_path}")
    
    return employees

# ========================
# 2. ATTENDANCE FUNCTIONS
# ========================

def initialize_attendance_file():
    """Create attendance file with headers if it doesn't exist"""
    if not os.path.exists("attendance.xlsx"):
        df = pd.DataFrame(columns=["Date", "RollNo", "Name", "Status"])
        df.to_excel("attendance.xlsx", index=False)

def mark_attendance(roll_no, name, status="Present"):
    """Mark attendance in Excel file"""
    today = datetime.now().strftime("%Y-%m-%d")
    
    try:
        df = pd.read_excel("attendance.xlsx")
    except FileNotFoundError:
        df = pd.DataFrame(columns=["Date", "RollNo", "Name", "Status"])
    
    # Check if attendance already marked today
    existing_entry = df[(df["Date"] == today) & (df["RollNo"] == roll_no)]
    
    if existing_entry.empty:
        new_entry = pd.DataFrame({
            "Date": [today],
            "RollNo": [roll_no],
            "Name": [name],
            "Status": [status]
        })
        df = pd.concat([df, new_entry], ignore_index=True)
        df.to_excel("attendance.xlsx", index=False)
        print(f"Attendance marked: {name} ({status})")

def mark_absent_employees(employees_db, present_roll_nos):
    """Mark employees who were not detected as absent"""
    today = datetime.now().strftime("%Y-%m-%d")
    
    for employee in employees_db:
        if employee["RollNo"] not in present_roll_nos:
            mark_attendance(employee["RollNo"], employee["Name"], "Absent")

# ========================
# 3. FACE RECOGNITION LOOP
# ========================

def run_attendance_system():
    # Initialize employee database
    create_employee_database()
    employees_db = load_employee_data()
    initialize_attendance_file()
    
    if not employees_db:
        print("Error: No valid employee data loaded. Check employees.csv and image files.")
        return
    
    # Initialize camera
    video_capture = cv2.VideoCapture(0)
    present_roll_nos = set()
    
    print("Starting attendance system. Press 'q' to quit...")
    
    while True:
        ret, frame = video_capture.read()
        if not ret:
            print("Error: Could not access camera")
            break
            
        # Convert to RGB (face_recognition uses RGB)
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # Detect faces
        face_locations = face_recognition.face_locations(rgb_frame)
        face_encodings = face_recognition.face_encodings(rgb_frame, face_locations)
        
        for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
            # Compare with known faces
            matches = face_recognition.compare_faces(
                [employee["Encoding"] for employee in employees_db],
                face_encoding,
                tolerance=0.6
            )
            
            name = "Unknown"
            roll_no = "N/A"
            
            if True in matches:
                matched_idx = matches.index(True)
                employee = employees_db[matched_idx]
                name = employee["Name"]
                roll_no = employee["RollNo"]
                
                if roll_no not in present_roll_nos:
                    mark_attendance(roll_no, name)
                    present_roll_nos.add(roll_no)
            
            # Draw bounding box and label
            cv2.rectangle(frame, (left, top), (right, bottom), (0, 255, 0), 2)
            cv2.putText(frame, f"{name} (ID: {roll_no})", 
                        (left, top - 10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        
        cv2.imshow("Attendance System", frame)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    # Before exiting, mark absent employees
    mark_absent_employees(employees_db, present_roll_nos)
    
    video_capture.release()
    cv2.destroyAllWindows()
    print("Attendance system stopped. Data saved to attendance.xlsx")

# ========================
# RUN THE SYSTEM
# ========================

if __name__ == "__main__":
    run_attendance_system()

Created employees.csv with sample data
Starting attendance system. Press 'q' to quit...
Attendance marked: Simret Mequanent (Present)
Attendance marked: Milkias Elias (Present)
Attendance marked: Kibru Bogale (Present)
Attendance marked: Mulatu Wondimu (Present)
Attendance marked: Tigist Teshome (Present)
